# Credit Card Customer Segmentation & Spend Analysis
### Exploratory Data Analysis (EDA)

In [3]:
import pandas as pd
df = pd.read_csv('dim_customers.csv')

In [4]:
df.head()

,customer_id,age_group,city,occupation,gender,marital status,avg_income
0,ATQCUS1825,45+,Bengaluru,Salaried IT Employees,Male,Married,73523
1,ATQCUS0809,25-34,Hyderabad,Salaried Other Employees,Male,Married,39922
2,ATQCUS0663,25-34,Chennai,Salaried Other Employees,Male,Married,37702
3,ATQCUS0452,25-34,Delhi NCR,Government Employees,Male,Married,54090
4,ATQCUS3350,21-24,Bengaluru,Freelancers,Male,Single,28376


In [7]:
df.describe(include="all")

,customer_id,age_group,city,occupation,gender,marital status,avg_income
count,4000,4000,4000,4000,4000,4000,4000.000000
unique,4000,4,5,5,2,2,NaN
top,ATQCUS1825,25-34,Mumbai,Salaried IT Employees,Male,Married,NaN
freq,1,1498,1078,1294,2597,3136,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,51657.032250
std,NaN,NaN,NaN,NaN,NaN,NaN,14690.140645
min,NaN,NaN,NaN,NaN,NaN,NaN,24816.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,38701.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,50422.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,64773.250000


In [9]:
df.isnull().sum()

customer_id       0
age_group         0
city              0
occupation        0
gender            0
marital status    0
avg_income        0
dtype: int64

## Data Understanding & Initial Exploration

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     4000 non-null   object
 1   age_group       4000 non-null   object
 2   city            4000 non-null   object
 3   occupation      4000 non-null   object
 4   gender          4000 non-null   object
 5   marital status  4000 non-null   object
 6   avg_income      4000 non-null   int64 
dtypes: int64(1), object(6)
memory usage: 218.9+ KB


In [12]:
df.shape

(4000, 7)

In [13]:
df = pd.read_csv('fact_spends.csv')

In [14]:
df.head()

,customer_id,month,category,payment_type,spend
0,ATQCUS1371,July,Health & Wellness,Credit Card,1114
1,ATQCUS0368,October,Groceries,Credit Card,1466
2,ATQCUS0595,May,Health & Wellness,Credit Card,387
3,ATQCUS0667,October,Electronics,Credit Card,1137
4,ATQCUS3477,September,Bills,UPI,2102


In [15]:
df.columns

Index(['customer_id', 'month', 'category', 'payment_type', 'spend'], dtype='object')

In [17]:
df.isnull().sum()

customer_id     0
month           0
category        0
payment_type    0
spend           0
dtype: int64

In [18]:
df.shape

(864000, 5)

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864000 entries, 0 to 863999
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   customer_id   864000 non-null  object
 1   month         864000 non-null  object
 2   category      864000 non-null  object
 3   payment_type  864000 non-null  object
 4   spend         864000 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 33.0+ MB


In [20]:
df_customers = pd.read_csv('dim_customers.csv')

In [21]:
df_spends = pd.read_csv('fact_spends.csv')

In [22]:
df_merged = df_customers.merge(df_spends, on="customer_id", how="inner")


In [23]:
df_merged.shape

(864000, 11)

In [24]:
# total spend per customer 
customer_spend = df_merged.groupby("customer_id")["spend"].sum().reset_index()
customer_spend.describe()

,spend
count,4000.000000
mean,132724.438750
std,54988.167095
min,35265.000000
25%,90933.750000
50%,120392.500000
75%,163112.500000
max,315201.000000


## Feature Engineering

In [25]:
# segmenting spender
customer_spend["segment"] = pd.qcut(
    customer_spend["spend"],
    q=3,
    labels=["Low", "Medium", "High"]
)

customer_spend["segment"].value_counts()

segment
Low       1334
Medium    1333
High      1333
Name: count, dtype: int64

In [26]:
# merging the segment to main data set 

df_merged = df_merged.merge(customer_spend[["customer_id", "segment"]], 
                            on="customer_id", 
                            how="left")

In [27]:
df_merged["segment"].value_counts()

segment
Low       288144
Medium    287928
High      287928
Name: count, dtype: int64

In [28]:
df_merged.groupby("segment")["spend"].sum().sort_values(ascending=False)

/var/folders/m8/d78rtfd57fx3mn3x8725xn280000gn/T/ipykernel_84134/3172919378.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_merged.groupby("segment")["spend"].sum().sort_values(ascending=False)


segment
High      263129213
Medium    161600316
Low       106168226
Name: spend, dtype: int64

In [ ]:
#insight -  High segment alone contributes the largest revenue share.

In [29]:
# revenue percentage contribution
segment_revenue = df_merged.groupby("segment")["spend"].sum()
segment_revenue / segment_revenue.sum() * 100

/var/folders/m8/d78rtfd57fx3mn3x8725xn280000gn/T/ipykernel_84134/2275790627.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segment_revenue = df_merged.groupby("segment")["spend"].sum()


segment
Low       19.997867
Medium    30.439066
High      49.563068
Name: spend, dtype: float64

In [31]:
df_merged.columns

Index(['customer_id', 'age_group', 'city', 'occupation', 'gender',
       'marital status', 'avg_income', 'month', 'category', 'payment_type',
       'spend', 'segment'],
      dtype='object')

In [33]:
#creating income  buckets

df_merged["income_segment"] = pd.qcut(
    df_merged["avg_income"],
    q=3,
    labels=["Low Income", "Mid Income", "High Income"]
)

In [34]:
df_merged[df_merged["segment"] == "High"] \
    .groupby("income_segment")["customer_id"] \
    .nunique()

/var/folders/m8/d78rtfd57fx3mn3x8725xn280000gn/T/ipykernel_84134/1644144964.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("income_segment")["customer_id"] \


income_segment
Low Income      16
Mid Income     353
High Income    964
Name: customer_id, dtype: int64

In [35]:
# avg spend per income segment 
df_merged.groupby("income_segment")["spend"].mean()

/var/folders/m8/d78rtfd57fx3mn3x8725xn280000gn/T/ipykernel_84134/381951333.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_merged.groupby("income_segment")["spend"].mean()


income_segment
Low Income     436.186143
Mid Income     573.092398
High Income    834.250184
Name: spend, dtype: float64

In [36]:
#spend by category for High segment 

df_merged[df_merged["segment"] == "High"] \
    .groupby("category")["spend"] \
    .sum() \
    .sort_values(ascending=False)

category
Bills                55099235
Groceries            44571427
Electronics          39767285
Health & Wellness    32480977
Travel               29710214
Food                 20749310
Entertainment        18247005
Apparel              14662669
Others                7841091
Name: spend, dtype: int64

In [37]:
#payment_type distrubution for high segment 

df_merged[df_merged["segment"] == "High"] \
    .groupby("payment_type")["spend"] \
    .sum() \
    .sort_values(ascending=False)

payment_type
Credit Card    110356037
UPI             66584558
Debit Card      59332651
Net Banking     26855967
Name: spend, dtype: int64

In [39]:
from sqlalchemy import create_engine

username = "postgres"
password = "Najil200913"
host = "localhost"
port = "5432"
database = "mitron_bank"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# Load customers table
df_customers.to_sql("customers", engine, if_exists="replace", index=False)

# Load transactions table
df_spends.to_sql("transactions", engine, if_exists="replace", index=False)

print("Tables successfully loaded.")

Tables successfully loaded.


## Key Insights

### 1. Revenue Contribution is Highly Skewed
A relatively small segment of high utilization customers contributes a disproportionately large share of total spend.  
This indicates that revenue is concentrated among heavy credit users.

---

### 2. Medium Utilization Segment is the Largest Opportunity
The majority of customers fall under the medium utilization category.  
These customers already show active spending behavior but have room to increase credit usage.

**Business Impact:**  
This segment is ideal for scalable growth strategies.

---

### 3. Salaried IT Professionals are High-Value Customers
Customers in IT and salaried roles demonstrate higher income stability and stronger spending behavior.

**Business Impact:**  
They represent the most reliable segment for credit card targeting.

---

### 4. Age Group 25–45 Shows Strong Financial Activity
Customers in this age range exhibit higher income levels and consistent spending patterns.

**Business Impact:**  
This group is financially mature and suitable for both premium and standard credit cards.

---

### 5. Metro Cities Drive Maximum Revenue
Cities like Mumbai and Delhi NCR contribute the highest overall spend.

**Business Impact:**  
Urban regions should be prioritized for acquisition campaigns.

---

### 6. Customers are Already Credit-Oriented
High usage of credit cards, UPI, and EMI indicates strong familiarity with credit-based transactions.

**Business Impact:**  
The opportunity lies in capturing existing behavior rather than building new habits.

---

## Conclusion

The analysis highlights clear customer segments based on utilization, income, and spending behavior.

- High utilization customers drive revenue  
- Medium segment drives growth  
- Certain demographics and cities outperform others  

These insights can be directly used to design targeted credit card strategies and improve customer acquisition efficiency.